# Demo: polynomial regression, closed form to Bayesian

Each section calls the problem's module in `src/` and shows its figures and key numbers; the
notebook has no logic of its own. `python -m src.run` produces the same numbers and figures as
files.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from src import p1, p2, p3, p4

np.set_printoptions(precision=4, suppress=False, linewidth=100)


def show(module, res):
    for fig in module.figures(res).values():
        display(fig)
        plt.close(fig)

## P1: closed form vs. Batch, Stochastic and Mini-batch GD

Degree 1 to 9 by least squares, then GD for d = 9 on the QR problem E(α) = ½‖Qα − t‖².

In [ ]:
r1 = p1.compute()
n1 = r1["numbers"]
show(p1, r1)
print(f"cond(Phi) = {n1['cond_phi']:.3g}, cond(Q) = {n1['cond_q']:.4f}, seed = {p1.SEED}")
for v, d in n1["verification"].items():
    print(
        f"{v:>9}: |w_GD - w_closed| = {d['abs_err']:.3g}  relative {d['rel_err']:.3g}  "
        f"E x{d['per_epoch_factor']:.4f} per epoch, < 1e-12 after {d['epochs_to_1e-12']} epochs"
    )
print(f"SGD vs. Batch, largest relative gap in E while E > 1e-10: {n1['sgd_batch_rel_gap']:.2g}")
for label, E in n1["E_100"].items():
    print(f"E after 100 epochs, {label}: {E:.3g}")
print("eta = 2.1, w_100 =", np.array(n1["w_100_too_large"]))
print("eta = 2.1, w_100 / w_closed =", round(n1["w_100_over_w_closed"], 1))

**Try it:** Q is 10 × 10 and orthogonal, so batch GD shrinks the error by (1 − η) per epoch.
η = 1 converges in one epoch; η = 2 oscillates; anything above 2 diverges. Change `eta`:

In [ ]:
from src.common.bases import monomial
from src.common.data import TRAIN, load

x, t = load(TRAIN)
Q, R = np.linalg.qr(monomial(x, p1.D))
eta = 1.0
alpha, losses = p1.gd(Q, t, eta, 5, "batch")
print("E per epoch:", losses)

## P2: ridge as the MAP estimate

In [ ]:
r2 = p2.compute()
n2 = r2["numbers"]
show(p2, r2)
g, flat, best = n2["gd"], n2["flat"], n2["panels"]["best"]
print(
    f"GD at lambda = {p2.GD_LAM:g}: {g['iterations']:,} iterations, "
    f"E~ = {g['objective']:.10f} (closed form {g['objective_closed']:.10f})"
)
print(
    f"cond(PhiT Phi) = {g['cond_PtP']:.3g} -> cond(PhiT Phi + lambda I) = {g['cond_PtP_lam']:.3g}"
)
print(f"lambda* = {best['lam']:.3g}, test RMSE {best['test_rmse']:.4f}")
print(
    f"within 1.05x: [{flat['lam_lo']:.3g}, {flat['lam_hi']:.3g}] ({flat['n_run']} grid points) "
    f"and {flat['others']}; peak {flat['peak_ratio']:.3f}x at lambda = {flat['peak_lam']:.3g}"
)
for name, p in n2["panels"].items():
    print(
        f"{name:>8}: lambda = {p['lam']:.3g}  |w| = {p['w_norm']:.4g}  "
        f"test RMSE {p['test_rmse']:.4f}"
    )

## P3: Lasso by subgradient descent (Legendre basis)

In [ ]:
r3 = p3.compute()
n3 = r3["numbers"]
show(p3, r3)
c, lasso = n3["check"], n3["table"]["lasso"]
print(
    f"lambda = {p3.CHECK_LAM}: E = {c['E_final']:.4f} (reference {p3.E_REF}), "
    f"|w - w*| = {c['w_minus_w_ref']:.3g}"
)
print("optimality:", c["optimality"])
print("w   =", np.array(c["w"]))
print("w*  =", p3.W_REF)
print(
    f"lambda* = {lasso['lam']:.3g}, test RMSE {lasso['test_rmse']:.4f}, "
    f"≈0 coefficients {n3['zero_idx_star']}"
)
for name, row in n3["table"].items():
    print(
        f"{name:>5}: lambda = {row['lam']:.3g}  test RMSE {row['test_rmse']:.4f}  "
        f"|w| = {row['w_norm']:.3f}  #|w_j| < 0.01: {row['n_small']}"
    )

## P4: Bayesian regression

In [ ]:
r4 = p4.compute()
n4 = r4["numbers"]
show(p4, r4)
print("m_N   =", np.array(n4["m_N"]))
print("w_MAP =", np.array(n4["w_map"]))
print(f"|w_MAP - m_N| = {n4['w_map_minus_m_N']:.2e}")
for size, w in n4["width"].items():
    print(
        f"N = {size:>2}: width mean {w['mean']:.3f}, max {w['max']:.3f}, "
        f"parameter share {100 * w['param_share']:.1f}%"
    )
print(f"noise floor {p4.FLOOR:.3f}")